# Stream processing: time, windows and state

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/concepts/stream-processing.ipynb)

These in-memory models show how event time, bounded state, and replay affect a streaming result.

## Setup and sample data

Orders carry event minutes and arrive out of order. Each row lists order ID and event minute.

In [ ]:
events = [(1, 2), (2, 8), (3, 3), (4, 16), (5, 4)]
print(events)

## Use event time to assign windows

Event time is when something happened. Processing time is when a worker sees it. A tumbling window assigns each event to one fixed interval.

**Where it matters**

Report orders in the interval they happened.

**Watch out for**

An arrival-time aggregate can shift delayed events into the wrong bucket.

In [ ]:
for order_id, minute in events:
    start = minute // 5 * 5
    print(order_id, (start, start + 5))

**What to notice:** Orders 1, 3 and 5 belong to the first window even though order 5 arrives last.

## Use a watermark to bound state

This teaching model keeps a running maximum event time and subtracts five minutes. It drops events for windows already closed and evicts their state.

**Where it matters**

Define a late-data policy and track dropped events.

**Watch out for**

Engines differ on when watermarks advance and how they treat late records. A watermark is a progress estimate, not proof that all events arrived.

In [ ]:
counts = {}
max_seen = 0
for order_id, minute in events:
    start = minute // 5 * 5
    if start + 5 <= max_seen - 5:
        print("late:", order_id)
        continue
    counts[start] = counts.get(start, 0) + 1
    max_seen = max(max_seen, minute)
    for closed in sorted(k for k in counts if k + 5 <= max_seen - 5):
        print("closed:", closed, counts.pop(closed))
print("open:", counts)

**What to notice:** The first two windows close after minute 16. Order 5 is late and only the 15–20 window remains open.

## Bound a stream join

A stream join holds unmatched input until matching records arrive or the allowed time range expires. A checkpoint records progress and state.

**Where it matters**

Match a payment to an order within a declared time interval.

**Watch out for**

Replaying from a checkpoint does not make an external sink idempotent. Keep a stable event ID or use a transactional sink.

In [ ]:
orders = [(10, 5), (11, 10)]
payments = [(10, 6), (11, 20)]
print([(order, paid_at) for order, placed_at in orders for paid_order, paid_at in payments
       if paid_order == order and placed_at <= paid_at < placed_at + 5])

**What to notice:** Order 10 matches in the five-minute interval. Order 11’s payment falls outside it.

## Check yourself

### Exercise 1: Apply the pattern

**Difficulty:** Easy · about 5 min

If the order 11 payment arrives at minute 14, should it match?

In [ ]:
# Try your solution here.

In [ ]:
print(10 <= 14 < 15)

## Key takeaways

- Report orders in the interval they happened.
- Define a late-data policy and track dropped events.
- Match a payment to an order within a declared time interval.

**Next:** [PySpark reference 14: Structured Streaming](../pyspark/pyspark-reference-14-structured-streaming.ipynb)